# ViMD Kaggle local: DI và SR không streaming

Notebook này tải dataset đã upload bằng `kagglehub`, đọc toàn bộ Parquet/Arrow thành `Dataset` thông thường và chạy các task DI/SR. Không sử dụng `streaming=True`.

Kết quả lưu tại `/kaggle/working/vimd_results.csv`.

## Thiết lập Kaggle

Nếu Kaggle báo thiếu package, bỏ comment cell cài đặt bên dưới. Model SR `wav2vec2-base-vi-vlsp2020` cần `pyctcdecode`/`kenlm` để dùng LM; notebook có fallback greedy CTC nếu decoder LM không tương thích.

In [ ]:
# Chỉ chạy một lần nếu kernel Kaggle chưa có dependencies.
# %pip install -q datasets transformers accelerate soundfile pyctcdecode
# %pip install -q https://github.com/kpu/kenlm/archive/master.zip

In [ ]:
from pathlib import Path
import csv
import io
import json
import random
import struct
import time
import unicodedata
import wave
from collections import Counter

import kagglehub
import numpy as np
from datasets import Audio, Dataset, DatasetDict, concatenate_datasets, load_dataset

DATASET_HANDLE = "thanhlamdev/vimd-dataset"
TARGET_SECONDS = 30 * 60
SEED = 20260916
RUN_SR = True
RUN_DI_TRAINING = False
RUN_DI_EVALUATION = True
SR_ACTIVE_MODELS = ["wav2vec2-base-vi-vlsp2020"]
DI_TASKS_TO_RUN = ["DI_VN_3", "DI_North", "DI_Central", "DI_South", "DI_VN_63"]
DI_BACKBONE = "nguyenvulebinh/wav2vec2-base-vi"
DI_TRAIN_STEPS = 500
DI_BATCH_SIZE = 4
DI_LEARNING_RATE = 3e-5
DI_EVAL_MAX_SAMPLES = 512
MAX_SR_SAMPLES = None

DATASET_ROOT = Path(kagglehub.dataset_download(DATASET_HANDLE))
OUTPUT_DIR = Path("/kaggle/working/vimd_experiment") if Path("/kaggle/working").exists() else Path.cwd() / "vimd_kaggle_experiment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR = OUTPUT_DIR / "sample_30min_audio"; SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT = OUTPUT_DIR / "di_checkpoints"
RESULTS_CSV = OUTPUT_DIR / "vimd_results.csv"

SR_MODEL_SPECS = {
    "wav2vec2-base-vietnamese": "dragonSwing/wav2vec2-base-vietnamese",
    "wav2vec2-base-vietnamese-160h": "khanhld/wav2vec2-base-vietnamese-160h",
    "wav2vec2-base-vietnamese-250h": "nguyenvulebinh/wav2vec2-base-vietnamese-250h",
    "wav2vec2-base-vi-vlsp2020": "nguyenvulebinh/wav2vec2-base-vi-vlsp2020",
    "whisper-base": "openai/whisper-base",
}
DI_TASKS = {
    "DI_VN_3": {"region": None, "label_column": "region", "labels": ["North", "Central", "South"]},
    "DI_North": {"region": "North", "label_column": "province_name", "labels": None},
    "DI_Central": {"region": "Central", "label_column": "province_name", "labels": None},
    "DI_South": {"region": "South", "label_column": "province_name", "labels": None},
    "DI_VN_63": {"region": None, "label_column": "province_name", "labels": None},
}
print("Dataset path:", DATASET_ROOT)
print("Files:", len(list(DATASET_ROOT.rglob("*"))))

## 1. Tìm file và load non-streaming

Hàm này hỗ trợ cả dataset Parquet và Arrow. Các file được nhóm theo tên `train`, `valid`/`validation`, `test`.

In [ ]:
def split_of(path):
    parts = [part.lower() for part in path.parts]
    name = path.name.lower()
    for split in ("train", "valid", "validation", "test"):
        if name.startswith(split + "-") or f"-{split}-" in name or split in parts:
            return "valid" if split == "validation" else split
    return None

def grouped_files(extension):
    groups = {"train": [], "valid": [], "test": []}
    for path in sorted(DATASET_ROOT.rglob(extension)):
        split = split_of(path)
        if split in groups: groups[split].append(str(path))
    return {split: files for split, files in groups.items() if files}

parquet_files = grouped_files("*.parquet")
arrow_files = grouped_files("*.arrow")
if arrow_files:
    print("Loading Arrow non-streaming:", {k: len(v) for k, v in arrow_files.items()})
    dataset = DatasetDict({split: concatenate_datasets([Dataset.from_file(path) for path in files]) for split, files in arrow_files.items()})
elif parquet_files:
    print("Loading Parquet non-streaming:", {k: len(v) for k, v in parquet_files.items()})
    dataset = load_dataset("parquet", data_files=parquet_files, streaming=False)
else:
    raise FileNotFoundError(f"Không tìm thấy Parquet/Arrow trong {DATASET_ROOT}")

dataset_no_decode = {split: part.cast_column("audio", Audio(decode=False)) for split, part in dataset.items()}
print("Rows:", {split: len(part) for split, part in dataset_no_decode.items()})
print("Columns:", dataset_no_decode[next(iter(dataset_no_decode))].column_names)

## 2. Audio preprocessing và sample 30 phút

In [ ]:
def audio_bytes(info):
    if info.get("bytes") is not None: return info["bytes"]
    path = info.get("path")
    if path and Path(path).exists(): return Path(path).read_bytes()
    raise FileNotFoundError(info)

def wav_encoding(data):
    marker = data.find(b"fmt ")
    if marker < 0: return "UNKNOWN"
    size = struct.unpack_from("<I", data, marker + 4)[0]; tag = struct.unpack_from("<H", data, marker + 8)[0]
    if tag == 1: return "PCM"
    if tag == 3: return "IEEE_FLOAT"
    if tag == 0xFFFE and size >= 40:
        subtype = struct.unpack_from("<H", data, marker + 32)[0]; return "IEEE_FLOAT" if subtype == 3 else ("PCM" if subtype == 1 else f"EXTENSIBLE_{subtype}")
    return f"FORMAT_{tag}"

def wav_metadata(data):
    try:
        with wave.open(io.BytesIO(data), "rb") as f: return {"sample_rate": f.getframerate(), "channels": f.getnchannels(), "sample_width": f.getsampwidth(), "encoding": wav_encoding(data), "frames": f.getnframes(), "duration_seconds": f.getnframes() / f.getframerate()}
    except wave.Error:
        import soundfile as sf
        info = sf.info(io.BytesIO(data)); return {"sample_rate": info.samplerate, "channels": info.channels, "sample_width": 4, "encoding": wav_encoding(data), "frames": info.frames, "duration_seconds": info.duration}

def wav_float32(data, target_rate=16000):
    if wav_encoding(data) == "IEEE_FLOAT":
        import soundfile as sf
        signal, source_rate = sf.read(io.BytesIO(data), dtype="float32", always_2d=False); signal = np.asarray(signal, dtype="float32")
        if signal.ndim > 1: signal = signal.mean(axis=1)
    else:
        with wave.open(io.BytesIO(data), "rb") as f: source_rate, channels, width = f.getframerate(), f.getnchannels(), f.getsampwidth(); frames = f.readframes(f.getnframes())
        if width == 2: signal = np.frombuffer(frames, dtype="<i2").astype("float32") / 32768.0
        elif width == 4: signal = np.frombuffer(frames, dtype="<i4").astype("float32") / 2147483648.0
        elif width == 1: signal = (np.frombuffer(frames, dtype="u1").astype("float32") - 128) / 128.0
        else: raise ValueError(f"Unsupported width={width}")
        if channels > 1: signal = signal.reshape(-1, channels).mean(axis=1)
    if source_rate != target_rate:
        target_length = max(1, round(len(signal) * target_rate / source_rate)); signal = np.interp(np.linspace(0, 1, target_length, endpoint=False), np.linspace(0, 1, len(signal), endpoint=False), signal).astype("float32")
    return signal, target_rate

refs = [(split, i) for split, part in dataset_no_decode.items() for i in range(len(part))]; random.Random(SEED).shuffle(refs)
sample_records, total_seconds = [], 0.0
for split, idx in refs:
    row = dataset_no_decode[split][idx]
    try: data = audio_bytes(row["audio"]); meta = wav_metadata(data)
    except Exception: continue
    name = row.get("filename") or f"row_{idx}.wav"; path = SAMPLE_DIR / f"{len(sample_records):05d}_{split}_{Path(name).name}"; path.write_bytes(data)
    sample_records.append({"split": split, "row_index": idx, "audio_path": str(path), "filename": name, "region": row.get("region"), "province_name": row.get("province_name"), "province_code": row.get("province_code"), "text": row.get("text"), **meta}); total_seconds += meta["duration_seconds"]
    if total_seconds >= TARGET_SECONDS: break
print("Sample count:", len(sample_records), "minutes:", total_seconds / 60)
print("Formats:", Counter((r["sample_rate"], r["channels"], r["sample_width"], r["encoding"]) for r in sample_records))
(OUTPUT_DIR / "sample_manifest.json").write_text(json.dumps({"records": sample_records, "total_seconds": total_seconds}, ensure_ascii=False, indent=2), encoding="utf-8")

## 3. Metrics và CSV

In [ ]:
def normalize_text(text):
    text = unicodedata.normalize("NFC", str(text or "").lower()); return " ".join("".join(ch if (ch.isalnum() or ch.isspace()) else " " for ch in text).split())
def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1): cur.append(min(cur[-1] + 1, prev[j] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]
def wer(ref, hyp):
    a, b = normalize_text(ref).split(), normalize_text(hyp).split(); return edit_distance(a, b) / max(1, len(a))
def cer(ref, hyp):
    a, b = normalize_text(ref).replace(" ", ""), normalize_text(hyp).replace(" ", ""); return edit_distance(list(a), list(b)) / max(1, len(a))
def macro_f1(true, pred, labels):
    scores = []
    for label in labels:
        tp = sum(a == label and b == label for a, b in zip(true, pred)); fp = sum(a != label and b == label for a, b in zip(true, pred)); fn = sum(a == label and b != label for a, b in zip(true, pred)); p = tp / (tp + fp) if tp + fp else 0; r = tp / (tp + fn) if tp + fn else 0; scores.append(2 * p * r / (p + r) if p + r else 0)
    return sum(scores) / max(1, len(scores))
RESULT_COLUMNS = ["record_type", "task", "model", "checkpoint", "split", "filename", "region", "province_name", "reference", "prediction", "correct", "wer", "cer", "f1_macro", "n_samples", "inference_seconds"]
result_rows = []
def write_results():
    with RESULTS_CSV.open("w", newline="", encoding="utf-8") as f: writer = csv.DictWriter(f, fieldnames=RESULT_COLUMNS, extrasaction="ignore"); writer.writeheader(); writer.writerows(result_rows)
    print("CSV:", RESULTS_CSV, "rows:", len(result_rows))

## 4. SR/ASR inference

Model được nạp trực tiếp, audio luôn được chuyển mono/16 kHz. Nếu LM decoder lỗi AddedToken/alphabet, notebook dùng greedy CTC.

In [ ]:
def load_sr(model_id):
    import torch
    from transformers import AutoModelForCTC, AutoProcessor
    if "vlsp2020" in model_id.lower():
        try:
            from transformers import Wav2Vec2ProcessorWithLM
            processor = Wav2Vec2ProcessorWithLM.from_pretrained(model_id); mode = "kenlm"
        except Exception as exc:
            print("LM decoder lỗi, dùng greedy CTC:", repr(exc))
            from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor
            processor = Wav2Vec2Processor(feature_extractor=Wav2Vec2FeatureExtractor.from_pretrained(model_id), tokenizer=Wav2Vec2CTCTokenizer.from_pretrained(model_id)); mode = "greedy"
    else:
        processor = AutoProcessor.from_pretrained(model_id); mode = "default"
    if "whisper" in model_id.lower():
        from transformers import AutoModelForSpeechSeq2Seq
        model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id); kind = "seq2seq"
    else:
        model = AutoModelForCTC.from_pretrained(model_id); kind = "ctc"
    device = "cuda" if torch.cuda.is_available() else "cpu"; model.to(device).eval(); return processor, model, kind, device, mode

def transcribe(path, processor, model, kind, device, mode):
    import torch
    signal, rate = wav_float32(Path(path).read_bytes(), 16000); inputs = processor(signal, sampling_rate=rate, return_tensors="pt", padding=True); inputs = {k: v.to(device) for k, v in inputs.items() if hasattr(v, "to")}
    with torch.no_grad():
        if kind == "ctc":
            logits = model(**inputs).logits
            if mode == "kenlm":
                try: return processor.decode(logits[0].cpu().numpy(), beam_width=100).text.strip()
                except Exception: pass
            return processor.batch_decode(logits.argmax(-1), skip_special_tokens=True)[0].strip()
        return processor.batch_decode(model.generate(**inputs, max_new_tokens=256), skip_special_tokens=True)[0].strip()

if RUN_SR:
    for model_name in SR_ACTIVE_MODELS:
        model_id = SR_MODEL_SPECS[model_name]
        try:
            processor, model, kind, device, mode = load_sr(model_id); selected = sample_records if MAX_SR_SAMPLES is None else sample_records[:MAX_SR_SAMPLES]; scored = []
            for record in selected:
                start = time.perf_counter()
                try:
                    prediction = transcribe(record["audio_path"], processor, model, kind, device, mode); scored.append({"record_type": "sample", "task": "SR", "model": model_name, "checkpoint": model_id, "split": record["split"], "filename": record["filename"], "region": record["region"], "province_name": record["province_name"], "reference": record["text"], "prediction": prediction, "wer": wer(record["text"], prediction), "cer": cer(record["text"], prediction), "inference_seconds": time.perf_counter() - start})
                except Exception as exc: scored.append({"record_type": "error", "task": "SR", "model": model_name, "checkpoint": model_id, "filename": record["filename"], "prediction": repr(exc)})
            result_rows.extend(scored); valid = [r for r in scored if r["record_type"] == "sample"]
            if valid: result_rows.append({"record_type": "aggregate", "task": "SR", "model": model_name, "checkpoint": model_id, "wer": sum(r["wer"] for r in valid) / len(valid), "cer": sum(r["cer"] for r in valid) / len(valid), "n_samples": len(valid)})
            write_results()
        except Exception as exc: print("Không load được SR model:", repr(exc))
else: print("RUN_SR=False")

## 5. DI checkpoint và macro-F1

In [ ]:
def accepts(row, task):
    region = DI_TASKS[task]["region"]; return region is None or row.get("region") == region
def labels_for(task):
    if DI_TASKS[task]["labels"] is not None: return list(DI_TASKS[task]["labels"])
    values = set()
    for stream in dataset_no_decode.values():
        for row in stream.select_columns(["region", "province_name"]):
            if accepts(row, task) and row.get("province_name"): values.add(row["province_name"])
    return sorted(values)
def make_collator(extractor):
    import torch
    def collate(features):
        batch = extractor.pad([{"input_values": f["input_values"]} for f in features], padding=True, return_tensors="pt"); batch["labels"] = torch.tensor([f["label"] for f in features], dtype=torch.long); return batch
    return collate
def train_di(task):
    import torch
    from torch.utils.data import DataLoader
    from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
    labels = labels_for(task); label2id = {x: i for i, x in enumerate(labels)}; id2label = {i: x for x, i in label2id.items()}; extractor = AutoFeatureExtractor.from_pretrained(DI_BACKBONE)
    model = AutoModelForAudioClassification.from_pretrained(DI_BACKBONE, num_labels=len(labels), label2id=label2id, id2label=id2label, ignore_mismatched_sizes=True); device = "cuda" if torch.cuda.is_available() else "cpu"; model.to(device).train()
    def preprocess(row): signal, rate = wav_float32(audio_bytes(row["audio"]), 16000); return {"input_values": signal, "label": label2id[row[DI_TASKS[task]["label_column"]]]}
    train_set = dataset_no_decode["train"].filter(lambda row: accepts(row, task)).shuffle(seed=SEED).map(preprocess); loader = DataLoader(train_set, batch_size=DI_BATCH_SIZE, collate_fn=make_collator(extractor), num_workers=0); optimizer = torch.optim.AdamW(model.parameters(), lr=DI_LEARNING_RATE)
    for step, batch in enumerate(loader, 1):
        batch = {k: v.to(device) for k, v in batch.items()}; loss = model(**batch).loss; loss.backward(); optimizer.step(); optimizer.zero_grad(set_to_none=True)
        if step == 1 or step % 25 == 0: print(task, step, float(loss.detach().cpu()))
        if step >= DI_TRAIN_STEPS: break
    checkpoint = CHECKPOINT_ROOT / task; checkpoint.mkdir(parents=True, exist_ok=True); model.save_pretrained(checkpoint); extractor.save_pretrained(checkpoint); (checkpoint / "labels.json").write_text(json.dumps(labels, ensure_ascii=False, indent=2), encoding="utf-8"); return checkpoint, labels

trained = {}
if RUN_DI_TRAINING:
    for task in DI_TASKS_TO_RUN: trained[task] = train_di(task)
else: print("RUN_DI_TRAINING=False")

In [ ]:
def evaluate_di(task, checkpoint, labels):
    import torch
    from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
    extractor = AutoFeatureExtractor.from_pretrained(checkpoint); model = AutoModelForAudioClassification.from_pretrained(checkpoint); device = "cuda" if torch.cuda.is_available() else "cpu"; model.to(device).eval(); true, pred, rows = [], [], []
    for row in dataset_no_decode["test"]:
        if not accepts(row, task) or len(true) >= DI_EVAL_MAX_SAMPLES: continue
        signal, rate = wav_float32(audio_bytes(row["audio"]), 16000); inputs = extractor(signal, sampling_rate=rate, return_tensors="pt"); inputs = {k: v.to(device) for k, v in inputs.items() if hasattr(v, "to")}
        with torch.no_grad(): predicted = labels[int(model(**inputs).logits.argmax(-1).item())]
        actual = row[DI_TASKS[task]["label_column"]]; true.append(actual); pred.append(predicted); rows.append({"record_type": "sample", "task": task, "model": "DI", "checkpoint": str(checkpoint), "split": "test", "filename": row.get("filename"), "region": row.get("region"), "province_name": row.get("province_name"), "reference": actual, "prediction": predicted, "correct": actual == predicted})
    score = macro_f1(true, pred, labels); rows.append({"record_type": "aggregate", "task": task, "model": "DI", "checkpoint": str(checkpoint), "f1_macro": score, "n_samples": len(true)}); result_rows.extend(rows); write_results(); print(task, "macro-F1=", score)

if RUN_DI_EVALUATION:
    for task in DI_TASKS_TO_RUN:
        checkpoint = CHECKPOINT_ROOT / task
        if task in trained: checkpoint, labels = trained[task]
        elif (checkpoint / "config.json").exists(): labels = json.loads((checkpoint / "labels.json").read_text(encoding="utf-8"))
        else: print("Bỏ qua", task, "- chưa có checkpoint"); continue
        evaluate_di(task, checkpoint, labels)
else: print("RUN_DI_EVALUATION=False")
write_results(); print("Hoàn tất:", RESULTS_CSV)

## Cách chạy

1. Attach dataset `thanhlamdev/vimd-dataset` vào Kaggle hoặc dùng đúng `DATASET_HANDLE`.
2. Chạy cell load dataset; kiểm tra `Rows` và `Columns`.
3. Chạy SR với `RUN_SR=True`; xem WER/CER trong CSV.
4. Muốn tạo DI checkpoint, đặt `RUN_DI_TRAINING=True` và bật GPU Kaggle.
5. Checkpoint lưu tại `/kaggle/working/vimd_experiment/di_checkpoints/<task>`.
6. CSV duy nhất: `/kaggle/working/vimd_experiment/vimd_results.csv`.